In [0]:
from pyspark.sql.types import *
from pyspark.sql import functions as F
import datetime as _dt

In [0]:
try:
    arrival_date = dbutils.widgets.get("arrival_date")
except Exception:
    arrival_date = _dt.date.today().strftime("%Y-%m-%d")

try:
    catalog = dbutils.widgets.get("catalog")
except Exception:
    catalog = "travel_bookings"
try:
    schema = dbutils.widgets.get("schema")
except Exception:
    schema = "default"

try:
    base_volume = dbutils.widgets.get("base_volume")
except Exception:
    base_volume = f"/Volumes/{catalog}/{schema}/data"
    
    

In [0]:
booking_path = f"{base_volume}/customer/customers_{arrival_date}.csv"
df = spark.read.format('csv').option("header","true").option("inferschema","true").load(booking_path)
df= df.withColumn("valid_from",F.to_date(F.lit(arrival_date))) \
      .withColumn("valid_to",F.to_date(F.lit("9999-12-31"))) \
      .withColumn("is_current",F.lit(True)) \
      .withColumn("business_date",F.to_date(F.lit(arrival_date)))

spark.sql(f"create schema if not exists {catalog}.bronze")
df.write.format("delta").mode("append").saveAsTable(f"{catalog}.bronze.customers_inc")

print(f"Ingested rows: {df.count()}")
    
